# TeleDrive v3.1 — Telegram → Google Drive (native Colab)

Run the cells top to bottom, once, in a single Colab runtime.

* Google Drive auth is **native Colab** — no desktop OAuth JSON upload and
  no pasted authorization code anywhere.
* Telegram API ID / API Hash are typed into a hidden prompt and are never
  printed, logged, snapshotted or packaged.
* The interface runs **locally in this runtime** (`share=False`). No public
  tunnel is created unless you deliberately opt in.
* The database and temporary files live on local `/content`; only the finished
  uploads go to Google Drive.

In [ ]:
# ==== Cell 1: restore the tested package and install pinned dependencies ====
# Mounted Drive is used ONLY to fetch the tested archive. SQLite, logs and temp
# files stay on local /content — never on the mounted Drive filesystem.
import os, pathlib, shutil, sys, zipfile

LOCAL_ROOT = pathlib.Path("/content")
PACKAGE_ZIP = LOCAL_ROOT / "teledrive_v3.1.zip"
DRIVE_ZIP = pathlib.Path("/content/drive/MyDrive/TeleDrive/teledrive_v3.1.zip")

try:
    from google.colab import drive as colab_drive
    colab_drive.mount("/content/drive", force_remount=False)
except Exception as exc:  # not on Colab, or the user declined the mount
    print("drive mount skipped:", type(exc).__name__)

if not PACKAGE_ZIP.exists() and DRIVE_ZIP.exists():
    shutil.copy2(DRIVE_ZIP, PACKAGE_ZIP)

assert PACKAGE_ZIP.exists(), (
    f"upload the tested archive to {PACKAGE_ZIP} (or {DRIVE_ZIP}) first"
)

with zipfile.ZipFile(PACKAGE_ZIP) as archive:
    archive.extractall(LOCAL_ROOT)

PACKAGE_DIR = next(p for p in LOCAL_ROOT.glob("teledrive-v3.1*") if p.is_dir())
os.chdir(PACKAGE_DIR)
sys.path.insert(0, str(PACKAGE_DIR))

# Exact pins, straight from the archive - requirements.lock is the ONE source of
# dependency truth. No version is ever hard-coded in this notebook.
!pip -q install -r "{PACKAGE_DIR}/requirements.lock"
print("dependency source:", PACKAGE_DIR / "requirements.lock")

print("package root:", PACKAGE_DIR)
print("runtime root (local, not Drive):", os.environ.setdefault(
    "TELEDRIVE_ROOT", "/content/teledrive_runtime"))

In [ ]:
# ==== Cell 2: bootstrap local directories, logging, SQLite migrations, WAL ====
import os
os.environ.setdefault("TELEDRIVE_ROOT", "/content/teledrive_runtime")

from teledrive import bootstrap

ctx = bootstrap.run()          # the ONE ApplicationContext for this runtime
print("schema version:", ctx.bootstrap_info["schema_version"])
print("free bytes on local disk:", ctx.bootstrap_info["free_bytes"])
print("journal mode:", ctx.db.journal_mode())

In [ ]:
# ==== Cell 3: credentials — hidden Telegram input + native Colab Drive auth ====
import getpass

# Telegram: hidden input, never echoed, never written to logs or snapshots.
api_id = getpass.getpass("Telegram API ID (hidden): ").strip()
api_hash = getpass.getpass("Telegram API Hash (hidden): ").strip()
assert api_id.isdigit() and api_hash, "API ID must be numeric and API Hash non-empty"

# Google Drive: native Colab credentials only. No desktop OAuth JSON upload,
# no pasted authorization code, no persisted Drive token file.
from google.colab import auth as colab_auth
import google.auth
from googleapiclient.discovery import build

colab_auth.authenticate_user(clear_output=False)
creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
drive_service = build("drive", "v3", credentials=creds, cache_discovery=False)

# The gate: nothing may report "Connected" before this call succeeds.
about = drive_service.about().get(
    fields="user(displayName,emailAddress),storageQuota(limit,usage)").execute()
print("drive verified for:", about["user"].get("emailAddress", "(hidden)"))

In [ ]:
# ==== Cell 4: inject into the ONE context and launch the interface ====
# No second context, no second event loop, no second Telegram client, no second
# Drive service. Everything below reuses the objects created in cells 2 and 3.
from teledrive.app import launch

ctx.telegram_auth.set_credentials(api_id, api_hash)   # secrets stay in memory
del api_id, api_hash

ctx.drive_auth.adopt_service(drive_service)           # already verified above
print("drive status:", ctx.drive_auth.status().state)

ctx.checkpoints.restore_and_reconcile()               # safe state, no auto-resume

# share=False: the UI is reachable inside this runtime only. A public link is an
# explicit opt-in (pass share to launch yourself) and is never the default.
# blocking=False: the cell returns immediately, so cells 5-7 (handoff, tests,
# maintenance) stay runnable while the interface keeps serving. The launch
# handle lives on ctx.ui and is closed by ctx.shutdown() in cell 7.
launch(ctx, share=False, inline=True, blocking=False)
print("ui running (non-blocking); cells 5-7 can be run while it serves")

In [ ]:
# ==== Cell 5: redacted handoff snapshot ====
from teledrive import handoff

# handoff.generate() runs every line through redaction before returning it.
print(handoff.generate(objective="controlled Colab run", phase="9 (Colab readiness)"))
print("snapshot generated (secrets redacted)")

In [ ]:
# ==== Cell 6: run the packaged test suite and fail loudly ====
import subprocess, sys

proc = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests"],
    capture_output=True, text=True,
)
print(proc.stdout)
print(proc.stderr, file=sys.stderr)
if proc.returncode != 0:
    raise SystemExit(f"test suite failed with exit code {proc.returncode}")
print("tests passed")

In [ ]:
# ==== Cell 7: safe maintenance — targeted cleanup, never a blind wipe ====
# There is deliberately no blind wipe of the temp directory here. Only files that
# belong to items verified as Uploaded are deleted; anything unrecognised or
# incomplete is moved to the quarantine directory for manual review.
print(ctx.checkpoints.persist())

from teledrive import storage_manager

report = storage_manager.cleanup_verified_temp()
print("deleted verified temp files:", report["deleted"])
print("quarantined unknown/incomplete files:", report["quarantined"])

ctx.shutdown()      # closes the UI handle, stops the async runtime, closes SQLite
print("runtime closed")